# Leaf Decomposition Imaging Pipeline

This notebook processes fluorescence and reflectance microscopy images of decomposing leaves
(species: *Betula*, *Ginkgo*; substrates: sand, carbonate, clay) and turns raw per-channel TIFFs
into calibrated composite images, leaf-level measurements, and QC/progression figures.

**What it does, in order:**

1. **File repair & scanning** — fixes mislabeled/GIF image files, ignores macOS `._` sidecar
   junk, and parses filenames (`species_substrate_rep_day_x<wavelength>_F<filter>_<exposure>ms.tif`)
   into structured metadata.
2. **Petri dish detection & scale calibration** — finds the dish rim in the reference channel
   and converts pixels → cm using the known 9 cm dish diameter, so a scale bar can be drawn on
   every image.
3. **Leaf segmentation** — separates the leaf from the background per image using a fallback
   chain (hysteresis thresholding at several sensitivities → Triangle thresholding → adaptive
   thresholding), with an optional GrabCut refinement pass. This handles low-contrast/barely
   fluorescing leaves that break a plain Otsu threshold.
4. **Canonical rotation** — rotates each *saved/displayed* image (never the raw pixels used for
   measurement) so the leaf's major axis is vertical, for easy visual before/after comparison.
5. **Composite building** — assembles RGB fluorescence and reflectance composites per sample,
   with a scale bar burned in.
6. **Measurements** — computes leaf surface area (cm²) and mean raw fluorescence per channel,
   always from the original unrotated pixels, written to a CSV.
7. **QC overlays & progression figures** — saves mask overlays (individually and in grids) so
   every automatic segmentation can be visually spot-checked, plus before/after progression
   figures per replicate.

**Note on experimental design:** each replicate has exactly one pre-burial (`initcond`) image
and one later time-point image (destructive sampling), not a full time series per replicate.

**Output layout** — everything is written as a sibling of the data folder (nothing is written
inside the data folder itself, so re-runs never scan their own output):
```
.../folder/fluorescence composites/
.../folder/reflection composites/
.../folder/leaf_mask_overlays/
.../folder/leaf_mask_overlay_grids/
.../folder/progression_figures/
.../folder/leaf_fluorescence_surface.csv
```

**Requirements:** `pip install tifffile numpy pandas Pillow opencv-python tqdm matplotlib scipy scikit-image`


In [ ]:
"""
Leaf decomposition imaging pipeline
====================================

This code was built to analyze and extract various elements from leaf 
imaging series:

  PART A - Composite building
      Builds fluorescence and reflectance RGB composite TIFFs for every
      (species, substrate, replicate, day) sample, with a scale bar
      added into each image (calibrated from the Petri dish rim visible
      in the x405/F4/10000ms channel), and assembles before/after
      progression figures.

  PART B - Leaf segmentation & measurement
      Detects the Petri dish, segments the leaf from the fluorescence
      channels, measures leaf surface area (cm^2) and mean raw
      fluorescence per channel, and writes a CSV plus quality-control
      overlay images so every automatic segmentation can be visually
      checked.

  PART C - Canonical rotation
      The leaf mask from Part B is reused in Part C: every composite
      image is rotated so the leaf's own major axis is vertical, with
      its bulkier end on the bottom and its narrower/tapering end on the
      right. This is a *per-image*, deterministic convention, it odes not 
      work systematically, but it gives a given leaf's "initcond" and "day N" 
      images end up comparably oriented, which is what makes the before/after 
      figures easy to read at a glance. Quantitative measurements (area, mean
      fluorescence) are always computed from the ORIGINAL, unrotated
      pixels, rotation (or any other transformation) only affects what gets 
      saved/displayed, never the numbers in the CSV.

How is leaf segmentation realized:
------------------------------------------------------------------
A single global Otsu threshold assumes a bimodal brightness histogram
(a clear "leaf" peak and a clear "background" peak). That mostly holds,
but breaks down for low-contrast samples -- e.g. a not fresh "initcond" 
leaf whose fluorescence barely rises above the background noise floor. 
In that case Otsu either finds no confident seed at all, or lumps
everything together into a region that gets rejected by the area
sanity-check. This version tries several strategies in order and uses
the first one that produces a plausible leaf-shaped region:

    1. Hysteresis thresholding (Canny-style two-level threshold) on the
       combined F5+F6 signal, at a few different sensitivity levels.
    2. Triangle thresholding (better suited than Otsu to skewed,
       near-unimodal histograms, exactly the low-contrast case above),
       tried on F6 then F5 individually.
    3. Local adaptive thresholding as a last resort, for cases with
       uneven illumination across the dish (not used on our dataset).

Each candidate mask still has to pass the same plausibility checks as
before (inside the dish, reasonable size) before being accepted, so this
adds recall without opening the door to false positives.

OUTPUT LAYOUT
-------------
Given a data directory. Everything is written into a single output folder 
that's a SIBLING of the data folder (so next to it, e.g. on the same USB 
stick), each kind of output in its own subfolder:
    .../Master project/fluorescence composites/
    .../Master project/reflection composites/
    .../Master project/leaf_mask_overlays/
    .../Master project/leaf_mask_overlay_grids/
    .../Master project/progression_figures/
    .../Master project/leaf_fluorescence_surface.csv
    .../Master project/_gif_originals_backup/      (only created if needed)

Nothing is ever written inside the data folder itself, so re-running the
pipeline never scans its own output.

Requirements
------------
    pip install tifffile numpy pandas Pillow opencv-python tqdm matplotlib scipy scikit-image
"""

# ======================================================================
# 1. IMPORTS
# ======================================================================

import re
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import tifffile
from PIL import Image
from tqdm import tqdm
from scipy import ndimage as ndi
from skimage.segmentation import watershed
import matplotlib.pyplot as plt


# ======================================================================
# 2. CONFIGURATION
# ======================================================================

# --- filename convention: species_substrate_rep_day_x<wl>_F<filt>_<exp>ms.tif
FNAME_RE = re.compile(
    r"^(?P<species>[A-Za-z]+)_"
    r"(?P<substrate>[A-Za-z]+)_"
    r"(?P<rep>rep\d+)_"
    r"(?P<day>[A-Za-z0-9]+)_"
    r"x(?P<wavelength>\d+)_"
    r"F(?P<filter>\d+)_"
    r"(?P<exposure>\d+)ms$",
    re.IGNORECASE,
)

# --- fluorescence composite channel map (all x405, distinguished by filter)
FLUO_CHANNELS = {
    "blue":  {"wavelength": 405, "filter": 4},
    "red":   {"wavelength": 405, "filter": 5},
    "green": {"wavelength": 405, "filter": 6},
}

# --- reflectance composite channel map (distinguished by wavelength + filter)
REFLEC_CHANNELS = {
    "blue":  {"wavelength": 470, "filter": 2},
    "green": {"wavelength": 525, "filter": 3},
    "red":   {"wavelength": 635, "filter": 5},
}

# --- Petri dish / scale bar ---
PETRI_DIAMETER_CM = 9.0
SCALE_BAR_CM = 1.0
# the channel used to detect the dish rim and calibrate pixels-per-cm
DISH_REFERENCE_WAVELENGTH = 405
DISH_REFERENCE_FILTER = 4

# --- leaf segmentation ---
DISH_INTERIOR_FRACTION = 0.96   # shrink dish mask slightly to exclude the rim itself
MIN_OBJECT_FRACTION = 0.005     # smallest plausible leaf, as a fraction of dish area
MAX_OBJECT_FRACTION = 0.60      # largest plausible leaf, as a fraction of dish area
MIN_LEAF_PIXELS = 100
MORPH_KERNEL_SIZE = 5
MORPH_CLOSE_ITERATIONS = 2
# hysteresis "low" threshold, as a fraction of the Otsu ("high") threshold.
# Tried in this order until one produces a plausible mask -- lower values
# are more inclusive (more recall, more risk of leaking into background).
HYSTERESIS_LOW_FRACTIONS = (0.35, 0.25, 0.15)

# --- optional GrabCut refinement pass (colour-based, for stubborn edges) ---
GRABCUT_REFINEMENT = True
GRABCUT_ITERATIONS = 5
GRABCUT_ERODE_PX = 8
GRABCUT_DILATE_PX = 25

# --- rotation ---
ROTATE_LEAVES_TO_CANONICAL_ORIENTATION = True # if FALSE won't rotate them

# --- GIF / mislabeled-file repair ---
GIF_BACKUP_DIRNAME = "_gif_originals_backup"

# --- progression figures ---
MAX_PAIRS_PER_BLOCK = 8   # before/after pairs per double-row block
PROGRESSION_DPI = 600     # sane DPI; images are shown with nearest-neighbour

# --- overlay grids ---
GRID_COLUMNS = 5
GRID_ROWS = 5
GRID_SIZE = GRID_COLUMNS * GRID_ROWS
GRID_DPI = 300

# --- Folders' names ---
OUTPUT_CSV_NAME = "leaf_fluorescence_surface.csv"
OVERLAY_FOLDER_NAME = "leaf_mask_overlays"
GRID_FOLDER_NAME = "leaf_mask_overlay_grids"


# ======================================================================
# 3. ENVIRONMENT HELPERS
# ======================================================================

def ask_for_data_directory() -> Path:
    """Interactively prompt for the data directory, retrying until a
    valid folder is given. Quotes pasted around the path are stripped
    automatically."""
    print("=" * 70)
    print("LEAF DECOMPOSITION IMAGING PIPELINE")
    print("=" * 70)
    print()

    while True:
        user_input = input("Data directory: ").strip().strip('"').strip("'")
        data_dir = Path(user_input).expanduser()
        if data_dir.exists() and data_dir.is_dir():
            print(f"\nUsing data directory:\n{data_dir}\n")
            return data_dir
        print(f"\nERROR: directory not found: {data_dir}\n")


# ======================================================================
# 4. FILENAME PARSING & FILE SCANNING
# ======================================================================

def is_macos_metadata_file(path: Path) -> bool:
    """macOS creates hidden '._filename' companion files on non-native
    filesystems (like a USB stick formatted for cross-platform use).
    They aren't real data and should be silently ignored."""
    return path.name.startswith("._")


def parse_filename(path: Path):
    """Parse one filename into its metadata fields, or return None if
    it doesn't match the expected naming convention."""
    match = FNAME_RE.match(path.stem)
    if match is None:
        return None
    meta = match.groupdict()
    meta["wavelength"] = int(meta["wavelength"])
    meta["filter"] = int(meta["filter"])
    meta["exposure"] = int(meta["exposure"])
    meta["path"] = path
    return meta


def day_sort_key(day: str):
    """Sort key so 'initcond' comes first, then '7d', '14d', ... in
    numeric order (not alphabetically, which would put '14d' before '7d')."""
    day = day.lower()
    if day == "initcond":
        return -1
    match = re.match(r"(\d+)d$", day)
    return int(match.group(1)) if match else 10_000


def format_day_value(day: str):
    """CSV-friendly day value: 'initcond' stays as-is, 'Nd' becomes the
    integer N (so the column can be sorted/plotted numerically)."""
    if day.lower() == "initcond":
        return "initcond"
    match = re.match(r"(\d+)d$", day.lower())
    return int(match.group(1)) if match else day


def scan_data_dir(data_dir: Path):
    """Walk the data directory and parse every TIFF file's metadata.
    Returns (dataframe, invalid_filename_list). Files already sitting
    inside our own output folders, or inside the GIF backup folder, are
    skipped so re-running the pipeline never scans its own output."""
    skip_dirnames = {
        "fluorescence composites", "reflection composites",
        "reflectance composites",  # tolerate the old folder name too
        "progression_figures", OVERLAY_FOLDER_NAME, GRID_FOLDER_NAME,
        GIF_BACKUP_DIRNAME.lower(),
    }

    records, invalid_files = [], []
    for path in tqdm(list(data_dir.rglob("*")), desc="Scanning files", unit="file"):
        if not path.is_file():
            continue
        if is_macos_metadata_file(path):
            continue
        if any(part.lower() in skip_dirnames for part in path.parts):
            continue
        if path.suffix.lower() not in (".tif", ".tiff"):
            continue

        meta = parse_filename(path)
        if meta is None:
            invalid_files.append(path)
        else:
            records.append(meta)

    return pd.DataFrame.from_records(records), invalid_files


def find_channel_file(group: pd.DataFrame, wavelength: int, filt: int):
    """Find the file in `group` matching a given wavelength + filter,
    preferring the longest exposure if duplicates exist."""
    matches = group[(group["wavelength"] == wavelength) & (group["filter"] == filt)]
    if len(matches) == 0:
        return None
    return matches.sort_values("exposure", ascending=False).iloc[0]["path"]


# ======================================================================
# 5. ROBUST IMAGE I/O handles GIF files, and .tif files 
# ======================================================================

def detect_real_format(path: Path):
    """Return the file's real image format by reading its bytes (e.g.
    'TIFF', 'GIF'), ignoring whatever the extension claims."""
    try:
        with Image.open(path) as img:
            return img.format
    except Exception:
        return None


def convert_gifs_to_tiff(data_dir: Path):
    """
    Repairs two related mistakes, backing up the original bytes (never
    deleting them) before rewriting a real TIFF in their place:

      (a) a file genuinely saved with a .gif extension
      (b) a file named .tif/.tiff whose actual encoded content is GIF
          (tifffile can't read these at all -- it checks content, not
          the extension)

    macOS AppleDouble sidecar files (hidden "._filename" companions
    macOS creates automatically on non-native filesystems, such as a
    USB stick formatted for cross-platform use) are skipped -- they
    aren't real images and can't be opened, so they'd otherwise show
    up as spurious "could not open file" warnings for every real file
    on the drive.

    Returns a list of (filename, message) warnings for anything that
    couldn't be repaired.
    """
    backup_dir = data_dir / GIF_BACKUP_DIRNAME
    warnings = []

    # (a) literal .gif files
    for gif_path in sorted(data_dir.rglob("*.gif")):
        if is_macos_metadata_file(gif_path):
            continue
        if backup_dir in gif_path.parents:
            continue
        if FNAME_RE.match(gif_path.stem) is None:
            warnings.append((gif_path.name, "gif doesn't match the naming pattern, skipped"))
            continue
        tif_path = gif_path.with_suffix(".tif")
        if tif_path.exists():
            warnings.append((gif_path.name, f"{tif_path.name} already exists, not overwritten"))
            continue
        arr = np.array(Image.open(gif_path).convert("L"), dtype=np.uint8)
        tifffile.imwrite(tif_path, arr)
        backup_dir.mkdir(parents=True, exist_ok=True)
        gif_path.rename(backup_dir / gif_path.name)

    # (b) .tif/.tiff files that are not actually TIFF-encoded
    for path in sorted(list(data_dir.rglob("*.tif")) + list(data_dir.rglob("*.tiff"))):
        if is_macos_metadata_file(path):
            continue
        if backup_dir in path.parents:
            continue
        real_format = detect_real_format(path)
        if real_format is None:
            warnings.append((path.name, "could not open file to check its real format"))
            continue
        if real_format.upper() == "TIFF":
            continue
        arr = np.array(Image.open(path).convert("L"), dtype=np.uint8)
        backup_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, backup_dir / f"{path.stem}_actually_{real_format.lower()}{path.suffix}")
        tifffile.imwrite(path, arr)  # overwrite in place, same filename

    return warnings


def robust_imread(path: Path) -> np.ndarray:
    """Read an image even if its extension lies about its format.
    tifffile is tried first (fast path); PIL is the fallback. This is a
    safety net -- convert_gifs_to_tiff() should already have fixed such
    files earlier in the pipeline."""
    try:
        return tifffile.imread(path)
    except Exception:
        return np.array(Image.open(path).convert("L"), dtype=np.uint8)


def get_single_channel(image: np.ndarray) -> np.ndarray:
    image = np.asarray(image)
    if image.ndim == 2:
        return image
    if image.ndim == 3:
        return image[..., 0]
    raise ValueError(f"Unsupported image shape: {image.shape}")


def normalise_to_uint8(image: np.ndarray, low_pct=1, high_pct=99.5) -> np.ndarray:
    """Percentile-stretch a single channel to 0-255, adaptively per image.
    This is deliberately used ONLY for segmentation/detection (dish
    detection, hysteresis thresholding, GrabCut's colour reference,
    etc.), where each image needs to be analysed on its own terms
    regardless of absolute brightness. It is NOT used for anything the
    person visually compares across samples -- see
    compute_global_display_bounds() / normalise_to_uint8_fixed() below
    for that."""
    image = get_single_channel(image).astype(np.float32)
    low, high = np.percentile(image, (low_pct, high_pct))
    if high <= low:
        high = low + 1
    return (np.clip((image - low) / (high - low), 0, 1) * 255).astype(np.uint8)


def normalise_channel_for_display(image: np.ndarray) -> np.ndarray:
    return normalise_to_uint8(image, low_pct=1, high_pct=99)


# ======================================================================
# GLOBAL DISPLAY CALIBRATION
# ======================================================================
#
# Anything the person actually look at to compare samples, progression
# figure panels, QC overlay backgrounds... must use the same brightness
# mapping for every image of a given channel. Per-image adaptive
# stretching (as segmentation uses, correctly, above) independently
# maximises each image's own contrast, which can make two genuinely
# different-brightness samples look nearly identical once displayed --
# hiding exactly the kind of discoloration/decay difference the person
# wants to be able to see by eye. The fix is a one-time calibration pass
# computing a FIXED percentile range per channel across the WHOLE
# dataset, used everywhere that channel is displayed from then on.
# (The saved composite TIFFs themselves are never touched by this...
# they keep raw, native pixel values either way.)

def compute_global_bounds(df: pd.DataFrame, wavelength: int, filt: int,
                           low_pct=1, high_pct=99, pixel_stride=4):
    """Global (lo, hi) display range for one (wavelength, filter) channel,
    computed from every matching file in the dataset. `pixel_stride`
    subsamples each image (every Nth pixel) to keep this pass fast on
    large datasets without needing the full-resolution data of every
    file in memory at once."""
    matches = df[(df["wavelength"] == wavelength) & (df["filter"] == filt)]
    samples = []
    for path in matches["path"]:
        try:
            img = get_single_channel(robust_imread(path)).astype(np.float32)
        except Exception:
            continue
        samples.append(img.flatten()[::pixel_stride])

    if not samples:
        return (0.0, 1.0)
    pooled = np.concatenate(samples)
    low, high = np.percentile(pooled, (low_pct, high_pct))
    if high <= low:
        high = low + 1
    return (float(low), float(high))


def compute_composite_display_bounds(df: pd.DataFrame, channel_map: dict):
    """channel_map is FLUO_CHANNELS or REFLEC_CHANNELS. Returns
    {'red': (lo, hi), 'green': (lo, hi), 'blue': (lo, hi)}, and also an
    ordered [R, G, B] list of the same, ready for to_display_uint8()."""
    bounds = {color: compute_global_bounds(df, spec["wavelength"], spec["filter"])
              for color, spec in channel_map.items()}
    ordered = [bounds["red"], bounds["green"], bounds["blue"]]
    return bounds, ordered


def normalise_to_uint8_fixed(image: np.ndarray, low: float, high: float) -> np.ndarray:
    """Like normalise_to_uint8, but with an externally-supplied FIXED
    range instead of computing one adaptively from this image alone --
    use this for anything the person will visually compare."""
    image = get_single_channel(image).astype(np.float32)
    if high <= low:
        high = low + 1
    return (np.clip((image - low) / (high - low), 0, 1) * 255).astype(np.uint8)


# ======================================================================
# 6. PETRI DISH DETECTION & SCALE CALIBRATION
# ======================================================================

def detect_petri_dish(image: np.ndarray):
    """Detect the dish rim with a Hough circle transform. Returns
    (center_x, center_y, radius_px), or None if no confident circle
    was found. Prefers large circles close to the image centre, since
    the dish is expected to roughly fill the frame."""
    gray = normalise_to_uint8(image)
    h, w = gray.shape
    min_dim = min(h, w)
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)

    circles = cv2.HoughCircles(
        blurred, cv2.HOUGH_GRADIENT, dp=1.2, minDist=min_dim // 2,
        param1=100, param2=30,
        minRadius=int(min_dim * 0.30), maxRadius=int(min_dim * 0.55),
    )
    if circles is None:
        return None

    circles = np.round(circles[0]).astype(int)
    cx0, cy0 = w / 2, h / 2
    # score = prefer large radius, penalise distance from image centre
    best = max(circles, key=lambda c: c[2] - 0.5 * np.hypot(c[0] - cx0, c[1] - cy0))
    return tuple(best)


def create_dish_mask(image_shape, dish_circle) -> np.ndarray:
    h, w = image_shape[:2]
    cx, cy, radius = dish_circle
    inner_radius = int(radius * DISH_INTERIOR_FRACTION)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mask, (cx, cy), inner_radius, 255, thickness=-1)
    return mask > 0


def calculate_pixels_per_cm(dish_circle, real_diameter_cm=PETRI_DIAMETER_CM) -> float:
    _, _, radius = dish_circle
    return (2 * radius) / real_diameter_cm


# ======================================================================
# 7. LEAF SEGMENTATION -- hysteresis thresholding with a fallback chain,
#    plus optional GrabCut colour refinement
# ======================================================================
#
# Otsu (a single global threshold) assumes a bimodal histogram: a clean
# "leaf" peak and a clean "background" peak. That mostly holds, but two
# things go wrong when it doesn't:
#   - the fainter, gradually-fading edges of a decomposing leaf sit
#     *below* whatever cutoff Otsu picks, so the mask misses real leaf
#     tissue (under-segmentation);
#   - a very low-contrast sample (e.g. a not fresh, barely-fluorescing
#     leaf) can break Otsu's bimodal assumption entirely, causing it
#     to find no seed region at all, or one so large/small it fails 
#     the area sanity-check.
#
# The functions below address both: hysteresis thresholding recovers
# faint-but-connected edges without picking up disconnected background
# noise, and a chain of fallback strategies (looser hysteresis, then
# Triangle thresholding, then adaptive local thresholding) handles the
# low-contrast case that defeats Otsu outright.

def combined_fluorescence_signal(image_a: np.ndarray, image_b: np.ndarray) -> np.ndarray:
    """Pixel-wise maximum of two normalised channels. A pixel only
    needs to show up strongly in ONE of the two channels to register --
    important because locally decomposed tissue can go dim in one
    channel while still showing signal in the other."""
    a = normalise_to_uint8(image_a).astype(np.float32)
    b = normalise_to_uint8(image_b).astype(np.float32)
    return np.maximum(a, b).astype(np.uint8)


def hysteresis_threshold(signal: np.ndarray, dish_mask: np.ndarray, low_fraction: float):
    """
    Two-level thresholding restricted to the dish interior:
      - seed_mask : pixels above the Otsu ("high") threshold
      - candidate : pixels above low_fraction * Otsu threshold
    Only candidate-mask connected components that contain at least one
    seed pixel are kept, this grows the confident core out through
    faint edges while rejecting patches of faint pixels with no strong
    seed nearby.
    """
    masked = signal.copy()
    masked[~dish_mask] = 0

    otsu_val, _ = cv2.threshold(masked[dish_mask], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    seed_mask = (masked > otsu_val) & dish_mask
    candidate_mask = (masked > otsu_val * low_fraction) & dish_mask

    n_labels, labels = cv2.connectedComponents(candidate_mask.astype(np.uint8))
    seed_labels = set(np.unique(labels[seed_mask])) - {0}
    if not seed_labels:
        return np.zeros_like(dish_mask)
    return np.isin(labels, list(seed_labels))


def clean_binary_mask(mask: np.ndarray) -> np.ndarray:
    """Close small gaps and fill fully-enclosed holes. Deliberately does
    NOT erode (no MORPH_OPEN) so faint/thin true leaf structures aren't
    stripped away, small noise blobs are rejected later by area instead, 
    which doesn't cost any real signal."""
    mask_u8 = mask.astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (MORPH_KERNEL_SIZE, MORPH_KERNEL_SIZE))
    mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, kernel, iterations=MORPH_CLOSE_ITERATIONS)

    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(mask_u8)
    cv2.drawContours(filled, contours, -1, 255, thickness=-1)
    return filled > 0


def select_leaf_component(binary_mask: np.ndarray, dish_mask: np.ndarray):
    """
    Clean the mask, then keep the largest connected component whose
    area is plausible for a leaf (not dust-speck small, not
    whole-dish large). Returns None if nothing plausible is found.

    If the cleaned mask's only component is implausibly large (rather
    than absent), that usually means noise or an illumination artifact
    formed a thin "bridge" connecting the true leaf region to the rest
    of the dish, so hysteresis grew across the whole thing. Erosion
    alone would separate the two, but returning the eroded result
    directly would permanently shrink the leaf and could clip off a
    genuinely thin structure such as the petiole/stem. Instead this
    uses the eroded components only as markers for a watershed split
    of the original (unshrunken) mask: watershed cuts precisely at the
    bridge's narrowest point and hands back each piece at its true
    full extent, so nothing that's genuinely part of the leaf; stem
    included gets removed.
    """
    cleaned = clean_binary_mask(binary_mask & dish_mask)
    dish_area = np.sum(dish_mask)
    min_area = max(MIN_LEAF_PIXELS, int(dish_area * MIN_OBJECT_FRACTION))
    max_area = int(dish_area * MAX_OBJECT_FRACTION)

    def best_plausible_component(mask):
        n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
            mask.astype(np.uint8), connectivity=8)
        candidates = [
            (stats[i, cv2.CC_STAT_AREA], labels == i)
            for i in range(1, n_labels)
            if min_area <= stats[i, cv2.CC_STAT_AREA] <= max_area
        ]
        if not candidates:
            return None
        return max(candidates, key=lambda c: c[0])[1]

    result = best_plausible_component(cleaned)
    if result is not None:
        return result

    # Only worth attempting a split if the failure was "too big", not
    # "too small" or "absent" separating blobs can't create signal
    # that isn't there in the first place.
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        cleaned.astype(np.uint8), connectivity=8)
    if n_labels <= 1:
        return None  # nothing there at all
    if max(stats[i, cv2.CC_STAT_AREA] for i in range(1, n_labels)) <= max_area:
        return None  # genuinely too small/absent, splitting won't help

    distance = ndi.distance_transform_edt(cleaned)
    for erosion_px in (3, 5, 8, 12):
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erosion_px, erosion_px))
        eroded = cv2.erode(cleaned.astype(np.uint8) * 255, kernel) > 0
        n_markers, markers = cv2.connectedComponents(eroded.astype(np.uint8))
        if n_markers <= 2:
            continue  # erosion didn't separate anything at this strength yet

        # Watershed hands each eroded seed back its TRUE, un-eroded extent,
        # split from its neighbours at the narrowest connecting point.
        # Selection is based on SEED size (how much confidently-leaf
        # tissue survived erosion), not on the final split area a
        # perfectly ambiguous bridge can tip a few pixels either way
        # during the split, but the seed size reliably identifies which
        # piece was the real, substantial blob to begin with.
        split_labels = watershed(-distance, markers, mask=cleaned)
        marker_sizes = {lbl: np.sum(markers == lbl) for lbl in range(1, n_markers)}
        plausible = [
            (marker_sizes[lbl], split_labels == lbl)
            for lbl in range(1, n_markers)
            if min_area <= np.sum(split_labels == lbl) <= max_area
        ]
        if plausible:
            return max(plausible, key=lambda c: c[0])[1]

    return None


def segment_leaf_robust(f5_image, f6_image, dish_circle, image_shape):
    """
    Try several segmentation strategies in order, returning the first
    one that produces a plausible leaf mask, along with a short label
    identifying which strategy worked (recorded in the CSV so you can
    see at a glance which samples needed a fallback).

    Returns (mask_or_None, method_label).
    """
    dish_mask = create_dish_mask(image_shape, dish_circle)
    have_both = f5_image is not None and f6_image is not None

    # --- Strategy 1: hysteresis on combined F5+F6 (or whichever is available) ---
    if have_both:
        signal = combined_fluorescence_signal(f5_image, f6_image)
    elif f6_image is not None:
        signal = normalise_to_uint8(f6_image)
    elif f5_image is not None:
        signal = normalise_to_uint8(f5_image)
    else:
        return None, "no F5/F6 image available"

    signal = cv2.GaussianBlur(signal, (5, 5), 0)
    for low_fraction in HYSTERESIS_LOW_FRACTIONS:
        hyst_mask = hysteresis_threshold(signal, dish_mask, low_fraction)
        if hyst_mask.any():
            leaf = select_leaf_component(hyst_mask, dish_mask)
            if leaf is not None:
                return leaf, f"hysteresis(low={low_fraction})"

    # --- Strategy 2: Triangle thresholding, better suited to the
    #     low-contrast / near-unimodal histograms that break Otsu ---
    for image, name in ((f6_image, "F6"), (f5_image, "F5")):
        if image is None:
            continue
        gray = normalise_to_uint8(image)
        gray_masked = gray.copy()
        gray_masked[~dish_mask] = 0
        _, tri = cv2.threshold(gray_masked, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_TRIANGLE)
        leaf = select_leaf_component((tri > 0) & dish_mask, dish_mask)
        if leaf is not None:
            return leaf, f"triangle({name})"

    # --- Strategy 3: local adaptive thresholding, for uneven illumination ---
    for image, name in ((f6_image, "F6"), (f5_image, "F5")):
        if image is None:
            continue
        gray = normalise_to_uint8(image)
        block_size = max(11, (min(image_shape) // 8) | 1)  # must be odd
        adaptive = cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY,
            block_size, -5)
        leaf = select_leaf_component((adaptive > 0) & dish_mask, dish_mask)
        if leaf is not None:
            return leaf, f"adaptive({name})"

    return None, "all segmentation strategies failed"


def refine_mask_with_grabcut(reflection_rgb, leaf_mask, dish_mask):
    """
    Second-pass refinement using GrabCut on the reflectance composite,
    which uses COLOUR statistics (not just brightness) to pull in
    ambiguous boundary pixels. Seeded conservatively from the mask we
    already trust, so it can only refine the boundary, not invent a
    new region. Returns the original mask unchanged if refinement isn't
    possible or doesn't look trustworthy (e.g. area balloons implausibly).
    """
    if reflection_rgb is None or not GRABCUT_REFINEMENT:
        return leaf_mask

    try:
        rgb = np.ascontiguousarray(reflection_rgb[..., :3]).astype(np.uint8)
        h, w = leaf_mask.shape
        if rgb.shape[:2] != (h, w):
            rgb = cv2.resize(rgb, (w, h), interpolation=cv2.INTER_LINEAR)

        kernel_fg = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (GRABCUT_ERODE_PX,) * 2)
        kernel_bg = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (GRABCUT_DILATE_PX,) * 2)
        sure_fg = cv2.erode(leaf_mask.astype(np.uint8), kernel_fg) > 0
        dilated = cv2.dilate(leaf_mask.astype(np.uint8), kernel_bg) > 0
        sure_bg = dish_mask & ~dilated

        if not sure_fg.any() or not sure_bg.any():
            return leaf_mask  # not enough seed diversity to refine safely

        gc_mask = np.full(leaf_mask.shape, cv2.GC_PR_BGD, dtype=np.uint8)
        gc_mask[sure_bg] = cv2.GC_BGD
        gc_mask[leaf_mask] = cv2.GC_PR_FGD
        gc_mask[sure_fg] = cv2.GC_FGD

        bgd_model, fgd_model = np.zeros((1, 65)), np.zeros((1, 65))
        cv2.grabCut(rgb, gc_mask, None, bgd_model, fgd_model,
                    GRABCUT_ITERATIONS, cv2.GC_INIT_WITH_MASK)

        refined = ((gc_mask == cv2.GC_FGD) | (gc_mask == cv2.GC_PR_FGD)) & dish_mask
        refined = clean_binary_mask(refined)

        if refined.sum() > 3 * max(leaf_mask.sum(), 1):
            return leaf_mask  # sanity check: refinement shouldn't balloon the area
        return refined
    except Exception:
        return leaf_mask


# ======================================================================
# 8. CANONICAL ROTATION
# ======================================================================
#
# Aligns a leaf's own major axis vertically, with its bulkier end on
# the bottom and its narrower/tapering end on the top, deterministically
# from its own mask. The exact sign conventions here were verified
# numerically against synthetic tapered shapes at a range of known
# angles before being used on real data.
#
# PAIR CONSISTENCY: which of the two 180-degree-apart options counts as
# "bulk-left" is decided from a skewness statistic, which is a mass
# distribution measure and decomposition can genuinely redistribute a
# leaf's mass enough to flip that statistic's sign between "initcond"
# and "day N" of the same leaf, even though both were independently
# computed "correctly" from what remains in each image. That's what
# produces mismatched top/bottom pairs. 

def _leaf_width_profile(mask: np.ndarray, n_bins: int = 24) -> np.ndarray:
    """A coarse shape signature: for a mask whose major axis is already
    horizontal, the (max-normalised) pixel count in each of n_bins equal
    slices along its horizontal extent: i.e. how wide the leaf is at
    each point along its length, left to right."""
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return np.zeros(n_bins)
    edges = np.linspace(xs.min(), xs.max() + 1, n_bins + 1)
    profile = np.array([np.sum((xs >= edges[i]) & (xs < edges[i + 1]))
                         for i in range(n_bins)], dtype=np.float64)
    peak = profile.max()
    return profile / peak if peak > 0 else profile


def _skew_based_flip(mask: np.ndarray, principal: np.ndarray, centered: np.ndarray) -> bool:
    """Fallback flip decision when there's no reference to match against
    (used for the first/only image of a leaf, typically 'initcond'):
    the longer, more-spread-out tail should end up on the right."""
    projection = centered @ principal
    std = projection.std()
    skew = np.mean(((projection - projection.mean()) / std) ** 3) if std > 0 else 0.0
    return skew < 0


def compute_canonical_angle(mask: np.ndarray, reference_profile: np.ndarray = None):
    """
    Angle (degrees) to rotate this image by so the mask's major axis
    becomes horizontal. Returns (angle_deg, width_profile) the
    profile is what a paired image should later be matched against via
    `reference_profile` to keep the pair's orientation consistent.

    If reference_profile is given, the left/right flip is chosen to
    best match it (pair-consistent). Otherwise it falls back to the
    skewness heuristic (used when there's nothing to match against yet).
    """
    ys, xs = np.nonzero(mask)
    pts = np.column_stack([xs, ys]).astype(np.float64)
    centered = pts - pts.mean(axis=0)

    cov = np.cov(centered.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    principal = eigvecs[:, np.argmax(eigvals)]
    angle_deg = np.degrees(np.arctan2(principal[1], principal[0]))

    rotated_normal = rotate_bound(mask.astype(np.uint8) * 255, angle_deg,
                                   interpolation=cv2.INTER_NEAREST) > 127
    profile_normal = _leaf_width_profile(rotated_normal)

    if reference_profile is not None and len(reference_profile) == len(profile_normal):
        profile_flipped = profile_normal[::-1]
        score_normal = -np.sum((profile_normal - reference_profile) ** 2)
        score_flipped = -np.sum((profile_flipped - reference_profile) ** 2)
        flip = score_flipped > score_normal
    else:
        flip = _skew_based_flip(mask, principal, centered)

    if flip:
        angle_deg += 180.0
        profile_normal = profile_normal[::-1]
    return angle_deg, profile_normal


def rotate_bound(image: np.ndarray, angle_deg: float, interpolation=cv2.INTER_LINEAR):
    """Rotate `image` by angle_deg around its centre, expanding the
    canvas so nothing gets cropped (rather than rotating in place,
    which can clip content near the edges). Works for 2D masks and
    3D multi-channel images alike; fill colour is 0 (black)."""
    h, w = image.shape[:2]
    cx, cy = w / 2.0, h / 2.0
    M = cv2.getRotationMatrix2D((cx, cy), angle_deg, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    new_w, new_h = int(h * sin + w * cos), int(h * cos + w * sin)
    M[0, 2] += new_w / 2 - cx
    M[1, 2] += new_h / 2 - cy
    return cv2.warpAffine(image, M, (new_w, new_h), flags=interpolation, borderValue=0)


# ======================================================================
# 9. COMPOSITE BUILDING (with scale bar and optional rotation)
# ======================================================================

def add_scale_bar(img: np.ndarray, pixels_per_cm, bar_cm: float = SCALE_BAR_CM) -> np.ndarray:
    """Add a scale bar (just the bar, no text label) into the bottom-left
    corner of a composite image. Returns the image unchanged if
    calibration wasn't available for this sample."""
    if pixels_per_cm is None:
        return img

    out = np.ascontiguousarray(img.copy())
    h, w = out.shape[:2]
    bar_len_px = int(round(bar_cm * pixels_per_cm))
    if bar_len_px <= 0:
        return out

    margin = max(20, int(min(h, w) * 0.04))
    thickness = max(4, int(min(h, w) * 0.008))
    x1, x2 = margin, min(w - margin, margin + bar_len_px)
    y = h - margin
    if x2 <= x1:
        return out

    max_val = np.iinfo(out.dtype).max if np.issubdtype(out.dtype, np.integer) else 1.0
    white, black = (max_val,) * 3, (0, 0, 0)

    cv2.line(out, (x1, y), (x2, y), black, thickness=thickness + 4, lineType=cv2.LINE_AA)
    cv2.line(out, (x1, y), (x2, y), white, thickness=thickness, lineType=cv2.LINE_AA)
    return out


def build_composite(channels: dict, pixels_per_cm=None, rotation_angle=None) -> np.ndarray:
    """
    Stack pre-loaded {'red':..., 'green':..., 'blue':...} channel
    arrays into an RGB composite. If rotation_angle is given, each
    channel is rotated first (same angle for all three, since they're
    the same physical capture). The scale bar is burned in AFTER
    rotation, so it always reads horizontally regardless of how the
    leaf itself was rotated.
    """
    if rotation_angle is not None:
        channels = {c: rotate_bound(arr, rotation_angle) for c, arr in channels.items()}

    dtype = channels["red"].dtype
    rgb = np.stack([channels["red"], channels["green"], channels["blue"]], axis=-1).astype(dtype)
    return add_scale_bar(rgb, pixels_per_cm)


def save_composite(rgb: np.ndarray, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(out_path, rgb, photometric="rgb")


# ======================================================================
# 10. MEASUREMENTS (always from the ORIGINAL, unrotated pixels)
# ======================================================================

def calculate_masked_mean(image: np.ndarray, mask: np.ndarray) -> float:
    image = get_single_channel(image)
    values = image[mask]
    return float(np.mean(values.astype(np.float64))) if values.size else np.nan


# ======================================================================
# 11. QUALITY-CONTROL OVERLAYS
# ======================================================================

def create_leaf_overlay(composite_rgb: np.ndarray, leaf_mask: np.ndarray) -> np.ndarray:
    """Overlay the leaf mask in translucent red on top of a (typically
    rotated) reflectance composite, for visual sanity-checking."""
    if composite_rgb.shape[:2] != leaf_mask.shape:
        composite_rgb = cv2.resize(composite_rgb, (leaf_mask.shape[1], leaf_mask.shape[0]),
                                    interpolation=cv2.INTER_LINEAR)
    overlay = composite_rgb.astype(np.float32)
    red_layer = np.zeros_like(overlay)
    red_layer[..., 0] = 255
    alpha = 0.45
    overlay[leaf_mask] = (1 - alpha) * overlay[leaf_mask] + alpha * red_layer[leaf_mask]
    return np.clip(overlay, 0, 255).astype(np.uint8)


def create_overlay_grids(overlay_records: list, grid_dir: Path):
    """Assemble individual QC overlays into 5x5 contact-sheet grids."""
    if not overlay_records:
        return
    grid_dir.mkdir(parents=True, exist_ok=True)
    n_grids = (len(overlay_records) + GRID_SIZE - 1) // GRID_SIZE

    for i in range(n_grids):
        records = overlay_records[i * GRID_SIZE: (i + 1) * GRID_SIZE]
        fig, axes = plt.subplots(GRID_ROWS, GRID_COLUMNS, figsize=(20, 20))
        axes = axes.flatten()
        for ax in axes:
            ax.axis("off")
        for ax, record in zip(axes, records):
            try:
                with Image.open(record["path"]) as img:
                    ax.imshow(np.asarray(img.convert("RGB")), interpolation="nearest")
            except Exception:
                ax.text(0.5, 0.5, "Could not load overlay", ha="center", va="center")
                continue
            ax.set_title(record["title"], fontsize=8)
            ax.axis("off")
        fig.tight_layout()
        fig.savefig(grid_dir / f"leaf_mask_grid_{i + 1:03d}.png", dpi=GRID_DPI, bbox_inches="tight")
        plt.close(fig)


# ======================================================================
# 12. PROGRESSION FIGURES
# ======================================================================

def to_display_uint8(img: np.ndarray, channel_bounds) -> np.ndarray:
    """Stretch each channel of an already-built RGB composite for
    display using fixed, externally-supplied (lo, hi) bounds per
    channel not an adaptive per-image percentile, which would
    independently maximise each image's own contrast and could hide
    genuine brightness/discoloration differences between samples. See
    compute_composite_display_bounds(). (The saved composite TIFFs keep
    their native bit depth regardless; this rescaling is display-only.)

    channel_bounds: [(lo, hi), (lo, hi), (lo, hi)] for channels 0, 1, 2.
    """
    out = np.zeros(img.shape[:2] + (3,), dtype=np.uint8)
    for c in range(3):
        lo, hi = channel_bounds[c]
        chan = img[..., c].astype(np.float32)
        out[..., c] = (np.clip((chan - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
    return out


def make_progression_figures(results: dict, fig_dir: Path, label: str, file_suffix: str,
                              channel_bounds):
    """
    One figure per (species, substrate) combination: every replicate
    that has both an 'initcond' and a later-day composite becomes one
    before/after column (day image on top, initial image below),
    ordered chronologically, up to MAX_PAIRS_PER_BLOCK columns per
    double-row block (extra pairs wrap onto additional blocks below).

    Uses nearest-neighbour interpolation in imshow and a sane DPI.

    channel_bounds is a FIXED [R, G, B] display range (see
    compute_composite_display_bounds), applied identically to every
    panel not an adaptive per-image stretch, so that brightness
    differences visible across the figure reflect real differences in
    the data rather than independent contrast normalisation.
    """
    fig_dir.mkdir(parents=True, exist_ok=True)

    by_combo = {}
    for (species, substrate, rep, day), path in results.items():
        by_combo.setdefault((species, substrate), {}).setdefault(rep, {})[day] = path

    for (species, substrate), rep_dict in tqdm(by_combo.items(),
                                                desc=f"Building {label} progression figures",
                                                unit="figure"):
        pairs = []
        for rep, days in rep_dict.items():
            if "initcond" not in days:
                continue
            init_path = days["initcond"]
            for day, path in days.items():
                if day != "initcond":
                    pairs.append((day, path, init_path, rep))

        if not pairs:
            continue
        pairs.sort(key=lambda x: (day_sort_key(x[0]), x[3]))

        n_pairs = len(pairs)
        n_blocks = (n_pairs + MAX_PAIRS_PER_BLOCK - 1) // MAX_PAIRS_PER_BLOCK
        n_rows, n_cols = n_blocks * 2, MAX_PAIRS_PER_BLOCK
        fig_width, fig_height = 18, max(7, 7.0 * n_blocks)

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height), squeeze=False)
        for row in range(n_rows):
            for col in range(n_cols):
                axes[row][col].axis("off")

        for idx, (day, day_path, init_path, rep) in enumerate(pairs):
            block = idx // MAX_PAIRS_PER_BLOCK
            col = idx % MAX_PAIRS_PER_BLOCK
            day_row, init_row = block * 2, block * 2 + 1

            day_img = tifffile.imread(day_path)
            init_img = tifffile.imread(init_path)

            axes[day_row][col].imshow(to_display_uint8(day_img, channel_bounds), interpolation="nearest")
            axes[day_row][col].set_title(f"{day} ({rep})", fontsize=13, pad=5)
            axes[day_row][col].axis("off")

            axes[init_row][col].imshow(to_display_uint8(init_img, channel_bounds), interpolation="nearest")
            axes[init_row][col].axis("off")

        for block in range(n_blocks):
            axes[block * 2][0].axis("on")
            axes[block * 2][0].set_ylabel("Day", rotation=0, ha="right", va="center", fontsize=12)
            axes[block * 2][0].set_xticks([])
            axes[block * 2][0].set_yticks([])
            for spine in axes[block * 2][0].spines.values():
                spine.set_visible(False)

            axes[block * 2 + 1][0].axis("on")
            axes[block * 2 + 1][0].set_ylabel("Initial", rotation=0, ha="right", va="center", fontsize=12)
            axes[block * 2 + 1][0].set_xticks([])
            axes[block * 2 + 1][0].set_yticks([])
            for spine in axes[block * 2 + 1][0].spines.values():
                spine.set_visible(False)

        fig.suptitle(f"{species} / {substrate} — {label} before/after", fontsize=14)
        fig.tight_layout(rect=[0, 0, 1, 0.96])
        fig.savefig(fig_dir / f"{species}_{substrate}_{file_suffix}.png",
                    dpi=PROGRESSION_DPI, bbox_inches="tight")
        plt.close(fig)


# ======================================================================
# 13. MAIN ORCHESTRATION -- one pass per sample covering both parts
# ======================================================================

def process_all_samples(df: pd.DataFrame, data_dir: Path, output_root: Path,
                         overlay_dir: Path, reflec_display_bounds):
    """
    For every (species, substrate, rep, day) sample:
      1. detect the Petri dish + scale from F4
      2. segment the leaf from F5+F6 (robust fallback chain + GrabCut)
      3. compute that sample's canonical rotation from the leaf mask,
         matched against the rep's initcond orientation for pair
         consistency
      4. build + save the (rotated) fluorescence and reflectance
         composites, each with a scale bar
      5. measure surface area and mean fluorescence from the ORIGINAL,
         unrotated pixels
      6. save a rotated QC overlay (leaf mask over reflectance composite)

    reflec_display_bounds is the fixed [R, G, B] range (see
    compute_composite_display_bounds) used only for the QC overlay
    background, so overlays stay visually comparable across samples
    it does not affect the saved composite TIFFs, which stay raw.

    Returns (csv_rows, fluo_results, reflec_results, errors, overlay_records)
    where *_results map (species, substrate, rep, day) -> saved composite
    path, in the same shape make_progression_figures() expects.
    """
    fluo_dir = output_root / "fluorescence composites"
    reflec_dir = output_root / "reflection composites"

    csv_rows, errors, overlay_records = [], [], []
    fluo_results, reflec_results = {}, {}
    # (species, substrate, rep) -> width profile from that rep's initcond
    # image, used to keep the paired day-N image's left/right flip
    # consistent (see compute_canonical_angle).
    rep_reference_profile = {}

    group_cols = ["species", "substrate", "rep", "day"]
    # Sort so each rep's 'initcond' sample is processed before its paired
    # day-N sample required so the reference profile exists by the
    # time it's needed. Plain groupby() sorts group keys alphabetically,
    # which would put e.g. '14d' before 'initcond' (digits sort before
    # letters), so day_sort_key is used explicitly instead.
    groups = sorted(df.groupby(group_cols), key=lambda kg: (kg[0][0], kg[0][1], kg[0][2],
                                                              day_sort_key(kg[0][3])))

    for keys, group in tqdm(groups, desc="Processing samples", unit="sample"):
        species, substrate, rep, day = keys
        sample_name = f"{species}_{substrate}_{rep}_{day}"

        # --- F4: Petri dish + scale ---
        f4_path = find_channel_file(group, DISH_REFERENCE_WAVELENGTH, DISH_REFERENCE_FILTER)
        if f4_path is None:
            errors.append((sample_name, "missing x405 F4 (dish/scale reference) image"))
            continue
        try:
            f4_image = robust_imread(f4_path)
        except Exception as exc:
            errors.append((sample_name, f"could not read F4: {exc}"))
            continue

        dish_circle = detect_petri_dish(f4_image)
        if dish_circle is None:
            errors.append((sample_name, "could not detect the Petri dish in F4"))
            continue
        pixels_per_cm = calculate_pixels_per_cm(dish_circle)
        dish_mask = create_dish_mask(f4_image.shape[:2], dish_circle)

        # --- F5/F6: leaf segmentation ---
        f5_fluo_path = find_channel_file(group, 405, FLUO_CHANNELS["red"]["filter"])
        f6_path = find_channel_file(group, 405, FLUO_CHANNELS["green"]["filter"])
        if f5_fluo_path is None:
            errors.append((sample_name, "missing x405 F5 image"))
            continue
        if f6_path is None:
            errors.append((sample_name, "missing x405 F6 image"))
            continue
        try:
            f5_image = robust_imread(f5_fluo_path)
            f6_image = robust_imread(f6_path)
        except Exception as exc:
            errors.append((sample_name, f"could not read F5/F6: {exc}"))
            continue

        leaf_mask, segmentation_method = segment_leaf_robust(
            f5_image, f6_image, dish_circle, f4_image.shape[:2])
        if leaf_mask is None:
            errors.append((sample_name, f"leaf could not be segmented ({segmentation_method})"))
            continue

        # --- build the reflectance composite (needed both for GrabCut
        #     refinement and as the QC overlay background) ---
        reflec_channels = {}
        reflec_ok = True
        for color, spec in REFLEC_CHANNELS.items():
            path = find_channel_file(group, spec["wavelength"], spec["filter"])
            if path is None:
                reflec_ok = False
                break
            try:
                reflec_channels[color] = robust_imread(path)
            except Exception:
                reflec_ok = False
                break

        reflec_rgb_original = None
        if reflec_ok:
            shapes = {c: a.shape for c, a in reflec_channels.items()}
            if len(set(shapes.values())) == 1:
                # Adaptive per-image normalisation is intentional here
                # this feeds GrabCut's internal colour modelling, not
                # anything the person visually compares, so maximising
                # each image's own contrast is the right choice.
                reflec_rgb_original = np.stack(
                    [normalise_channel_for_display(reflec_channels["red"]),
                     normalise_channel_for_display(reflec_channels["green"]),
                     normalise_channel_for_display(reflec_channels["blue"])], axis=-1)

        # --- optional GrabCut refinement (still in original orientation) ---
        if reflec_rgb_original is not None:
            refined = refine_mask_with_grabcut(reflec_rgb_original, leaf_mask, dish_mask)
            if refined.sum() != leaf_mask.sum():
                segmentation_method += "+grabcut"
            leaf_mask = refined

        # --- measurements: ALWAYS from the original, unrotated pixels ---
        surface_cm2 = leaf_mask.sum() / (pixels_per_cm ** 2)
        try:
            mean_f4 = calculate_masked_mean(f4_image, leaf_mask)
            mean_f5 = calculate_masked_mean(f5_image, leaf_mask)
            mean_f6 = calculate_masked_mean(f6_image, leaf_mask)
        except Exception as exc:
            errors.append((sample_name, f"fluorescence measurement failed: {exc}"))
            continue

        # --- additional filter measurements ---
        # IMPORTANT: filter number alone does not uniquely identify a channel;
        # the wavelength is part of the filename convention. Measure the
        # requested channels using the same leaf mask, from the original,
        # unrotated pixels. Missing channels are recorded as NaN.
        #
        # Channel mapping:
        #   F2 @ x470 nm
        #   F3 @ x525 nm
        #   F5 @ x635 nm
        filter_channels = {
            2: 470,
            3: 525,
            5: 635,
        }
        filter_means = {}
        for filt, wavelength in filter_channels.items():
            column_name = f"mean_value_F{filt}_x{wavelength}"
            filter_path = find_channel_file(group, wavelength, filt)
            if filter_path is None:
                filter_means[column_name] = np.nan
            else:
                try:
                    filter_image = robust_imread(filter_path)
                    filter_means[column_name] = calculate_masked_mean(
                        filter_image, leaf_mask
                    )
                except Exception as exc:
                    filter_means[column_name] = np.nan
                    errors.append(
                        (sample_name,
                         f"filter F{filt} @ x{wavelength} measurement failed: {exc}")
                    )

        csv_rows.append({
            "leaf": f"{species}_{rep}",
            "substrate": substrate,
            "date_in_days": format_day_value(day),
            "mean_value_F6_x405": mean_f6,
            # F5 @ x405 is the fluorescence red channel used by the existing
            # fluorescence pipeline; F5 @ x635 is the reflectance channel.
            "mean_value_F5_x405": mean_f5,
            "mean_value_F5_x635": filter_means["mean_value_F5_x635"],
            "mean_value_F4_x405": mean_f4,
            "mean_value_F3_x525": filter_means["mean_value_F3_x525"],
            "mean_value_F2_x470": filter_means["mean_value_F2_x470"],
            "surface_cm2": surface_cm2,
            "segmentation_method": segmentation_method,
        })

        # --- canonical rotation ---
        rep_key = (species, substrate, rep)
        if ROTATE_LEAVES_TO_CANONICAL_ORIENTATION:
            reference_profile = rep_reference_profile.get(rep_key)
            rotation_angle, leaf_profile = compute_canonical_angle(leaf_mask, reference_profile)
            if day.lower() == "initcond":
                rep_reference_profile[rep_key] = leaf_profile
        else:
            rotation_angle = None

        # --- fluorescence composite (rotated) ---
        fluo_channels = {"blue": f4_image, "red": f5_image, "green": f6_image}
        fluo_rgb = build_composite(fluo_channels, pixels_per_cm, rotation_angle)
        fluo_out = fluo_dir / f"{sample_name}_fluo_composite.tif"
        save_composite(fluo_rgb, fluo_out)
        fluo_results[keys] = fluo_out

        # --- reflectance composite (rotated) ---
        if reflec_rgb_original is not None:
            reflec_rgb = build_composite(reflec_channels, pixels_per_cm, rotation_angle)
            reflec_out = reflec_dir / f"{sample_name}_reflec_composite.tif"
            save_composite(reflec_rgb, reflec_out)
            reflec_results[keys] = reflec_out
        else:
            errors.append((sample_name, "missing reflectance channel(s), no reflectance composite saved"))

        # --- QC overlay: rotated mask over rotated reflectance composite ---
        if reflec_rgb_original is not None:
            rotated_mask = rotate_bound(leaf_mask.astype(np.uint8) * 255, rotation_angle or 0.0,
                                         interpolation=cv2.INTER_NEAREST) > 127
            rotated_reflec_display = rotate_bound(
                np.stack([normalise_to_uint8_fixed(reflec_channels["red"], *reflec_display_bounds[0]),
                          normalise_to_uint8_fixed(reflec_channels["green"], *reflec_display_bounds[1]),
                          normalise_to_uint8_fixed(reflec_channels["blue"], *reflec_display_bounds[2])], axis=-1),
                rotation_angle or 0.0)
            try:
                overlay = create_leaf_overlay(rotated_reflec_display, rotated_mask)
                overlay_path = overlay_dir / f"{sample_name}_leafmask.png"
                overlay_path.parent.mkdir(parents=True, exist_ok=True)
                Image.fromarray(overlay).save(overlay_path)
                overlay_records.append({
                    "path": overlay_path,
                    "title": f"{species} | {substrate}\n{rep} | {day}\n{segmentation_method}",
                })
            except Exception as exc:
                errors.append((sample_name, f"could not create QC overlay: {exc}"))

    return csv_rows, fluo_results, reflec_results, errors, overlay_records


def print_report(format_warnings, invalid_files, errors, n_success, n_overlays):
    if format_warnings:
        print("\n" + "=" * 70)
        print("FILE FORMAT REPAIRS / WARNINGS")
        print("=" * 70)
        for name, msg in format_warnings:
            print(f"  {name} -> {msg}")

    if invalid_files:
        print("\n" + "=" * 70)
        print("FILES WITH INVALID NAMES (ignored)")
        print("=" * 70)
        for path in invalid_files:
            print(f"  {path}")

    if errors:
        print("\n" + "=" * 70)
        print("PROCESSING ERRORS")
        print("=" * 70)
        for name, msg in errors:
            print(f"\n  {name}\n      -> {msg}")

    print("\n" + "=" * 70)
    print("FINISHED")
    print("=" * 70)
    print(f"\nSuccessfully processed samples: {n_success}")
    print(f"Invalid filenames: {len(invalid_files)}")
    print(f"Errors/warnings: {len(errors)}")
    print(f"QC overlays created: {n_overlays}\n")


# ======================================================================
# 14. MAIN
# ======================================================================

def main():
    data_dir = ask_for_data_directory()

    # Single output root, sibling of the data folder every kind of
    # output gets its own subfolder underneath it (see module docstring).
    output_root = data_dir.parent
    overlay_dir = output_root / OVERLAY_FOLDER_NAME
    grid_dir = output_root / GRID_FOLDER_NAME
    progression_dir = output_root / "progression_figures"
    csv_path = output_root / OUTPUT_CSV_NAME

    print("\nProcessing started...\n")

    format_warnings = convert_gifs_to_tiff(data_dir)

    df, invalid_files = scan_data_dir(data_dir)
    if df.empty:
        print_report(format_warnings, invalid_files, [], 0, 0)
        raise SystemExit("\nERROR: no valid TIFF files matching the naming pattern were found.\n")

    print("Calibrating display brightness ranges across the dataset ")
    _, fluo_display_bounds = compute_composite_display_bounds(df, FLUO_CHANNELS)
    reflec_display_bounds_dict, reflec_display_bounds = compute_composite_display_bounds(df, REFLEC_CHANNELS)

    csv_rows, fluo_results, reflec_results, errors, overlay_records = process_all_samples(
        df, data_dir, output_root, overlay_dir, reflec_display_bounds)

    # --- CSV ---
    columns = ["leaf", "substrate", "date_in_days", "mean_value_F6_x405",
               "mean_value_F5_x405", "mean_value_F5_x635", "mean_value_F4_x405",
               "mean_value_F3_x525", "mean_value_F2_x470", "mean_value_F1_x405",
               "surface_cm2", "segmentation_method"]
    pd.DataFrame(csv_rows, columns=columns).to_csv(csv_path, index=False)

    # --- QC overlay grids ---
    if overlay_records:
        print("\nCreating overlay grids...")
        create_overlay_grids(overlay_records, grid_dir)

    # --- progression figures (fixed display bounds, see calibration above,
    #     so brightness/colour differences across panels are real) ---
    print("\nBuilding progression figures...")
    make_progression_figures(fluo_results, progression_dir, "fluorescence", "fluo_progression",
                              fluo_display_bounds)
    make_progression_figures(reflec_results, progression_dir, "reflection", "reflection_progression",
                              reflec_display_bounds)

    print_report(format_warnings, invalid_files, errors, len(csv_rows), len(overlay_records))
    print("CSV:")
    print(f"  {csv_path}")
    print("\nProgression figures:")
    print(f"  {progression_dir}")
    print("\nFluorescence composites:")
    print(f"  {output_root / 'fluorescence composites'}")
    print("\nReflection composites:")
    print(f"  {output_root / 'reflection composites'}")
    print("\nQC overlays:")
    print(f"  {overlay_dir}")
    print("\nQC overlay grids:")
    print(f"  {grid_dir}\n")


if __name__ == "__main__":
    main()
